# DermaTriage — Bias Audit

Evaluate fairness across Fitzpatrick skin types for the trained models.
Compares the **teacher** (EfficientNet-B4) and the distilled **student**
(MobileNetV3-Small) per skin type, and computes the Deployment Viability Score
(DVS) for each variant.

Run from `ml_pipeline/notebooks/` after `make teacher` and `make distill`.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

sys.path.insert(0, os.path.abspath('..'))

from src.data.dataloader import build_dataloaders
from src.distillation.quantise import benchmark_inference
from src.evaluation.dvs import DVSInput, compute_dvs
from src.evaluation.fitzpatrick_eval import evaluate_fitzpatrick_stratified
from src.evaluation.metrics import compute_top1_accuracy
from src.models.student import build_student
from src.models.teacher import build_teacher
from src.utils.checkpoint import load_checkpoint

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = cfg['data']['num_classes']
print('Device:', device)

## Load model checkpoints

Adjust the checkpoint paths if your config overrides the defaults.

In [ ]:
teacher_ckpt = os.path.join('..', 'models', 'teacher', 'teacher_best.pt')
student_ckpt = os.path.join('..', 'models', 'student', 'student_best.pt')

teacher = build_teacher(num_classes=num_classes, pretrained=False).to(device)
load_checkpoint(teacher, None, teacher_ckpt, device)
teacher.eval()

student = build_student(num_classes=num_classes, pretrained=False).to(device)
load_checkpoint(student, None, student_ckpt, device)
student.eval()

print('Loaded teacher and student checkpoints.')

## Fitzpatrick-stratified evaluation on the test set

In [ ]:
_, _, test_loader = build_dataloaders(
    cfg, batch_size=cfg['distillation']['batch_size']
)

teacher_fitz = evaluate_fitzpatrick_stratified(teacher, test_loader, device)
student_fitz = evaluate_fitzpatrick_stratified(student, test_loader, device)

print('Teacher by Fitzpatrick:', {k: round(v, 4) for k, v in teacher_fitz.items()})
print('Student by Fitzpatrick:', {k: round(v, 4) for k, v in student_fitz.items()})

## Accuracy by Fitzpatrick type — teacher vs student

In [ ]:
types = sorted(set(teacher_fitz) | set(student_fitz))
x = np.arange(len(types))
width = 0.38

t_vals = [teacher_fitz.get(t, 0.0) for t in types]
s_vals = [student_fitz.get(t, 0.0) for t in types]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width / 2, t_vals, width, label='Teacher', color='#00695C')
ax.bar(x + width / 2, s_vals, width, label='Student', color='#FF8F00')
ax.set_xticks(x)
ax.set_xticklabels([f'Type {t}' for t in types])
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.set_title('Accuracy by Fitzpatrick skin type')
ax.legend()
plt.tight_layout()
plt.show()

## Deployment Viability Score per model variant

DVS combines overall accuracy, latency, model size and Fitzpatrick equity.

In [ ]:
weights = cfg['evaluation']['dvs_weights']
image_size = cfg['data']['image_size']


def state_size_mb(model):
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.pt') as tmp:
        torch.save(model.state_dict(), tmp.name)
        return os.path.getsize(tmp.name) / (1024 * 1024)


rows = []
for name, model, fitz in [('Teacher', teacher, teacher_fitz),
                          ('Student', student, student_fitz)]:
    top1 = compute_top1_accuracy(model, test_loader, device)
    latency_ms, _ = benchmark_inference(model, image_size=image_size, device='cpu')
    size_mb = state_size_mb(model)
    dvs = compute_dvs(
        DVSInput(
            top1_accuracy=top1,
            latency_ms=latency_ms,
            model_size_mb=size_mb,
            fitzpatrick_accuracies=fitz,
        ),
        weights,
    )
    rows.append((name, top1, latency_ms, size_mb, dvs))

print(f"{'Model':<10}{'Top-1':>8}{'Latency(ms)':>14}{'Size(MB)':>12}{'DVS':>8}")
for name, top1, lat, size, dvs in rows:
    print(f'{name:<10}{top1:>8.3f}{lat:>14.1f}{size:>12.2f}{dvs:>8.3f}')